# Phase B — dense visual event refinement
Visual-only, human-led review of the five Phase A target seeds. Raw images are read-only. The first-positive image bounds an event; it is not the exact biological event time.

1. Run the setup cells. They index only the three specified windows, prepare blank review tables, and save Stage 1 contact sheets.
2. Choose a seed below and launch Stage 1. Inspect ~90-second crops. Select `crack` or `radicle_emergence`. At a frame without a clear event, click **Mark last negative**. At a later clear-event frame, click **Mark first positive**. Click **Save reviewed interval**. Save optional frame states/notes separately.
3. Switch to Stage 2 and launch the viewer. It loads only the human-narrowed interval at native ~10-second spacing. Mark the final adjacent last-negative and first-positive frames, enter visual and identity confidence, then save.
4. Use **Save final figure** only after checking both boundary frames. The interval is `(last_negative, first_positive]`.

The ROI boxes come from Phase A. If the seed moves, edit the four ROI coordinate fields for that frame and click **Save ROI correction**; do not treat this as automatic tracking. If a widget fails, open the Stage 1 PNG contact sheet or generate the narrowed Stage 2 PNG using the fallback cell. Do not consult audio during this review.


In [ ]:
from pathlib import Path
import sys, subprocess, os
PROJECT = Path.cwd().resolve()
if PROJECT.name == 'notebooks': PROJECT = PROJECT.parent
if not (PROJECT / 'src' / 'dense_visual.py').exists():
    raise RuntimeError('Open this notebook inside the seed-germination project')
sys.path.insert(0, str(PROJECT))
from src.dense_visual import *
TABLES = PROJECT / 'results/tables/targeted_multimodal'
FIGURES = PROJECT / 'results/figures/targeted_multimodal'
RAW_ROOT = Path(os.environ.get("SEED_GERMINATION_SNAPSHOT_ROOT", PROJECT / "data/raw/snapshots"))
print('Python:', sys.executable)
print('Raw snapshot input:', RAW_ROOT)
assert RAW_ROOT.is_dir(), 'Mount the read-only INTENSO volume before proceeding'


In [ ]:
# Safe to rerun: existing human review tables are kept. Initial run scans only relevant groups.
subprocess.run([sys.executable, str(PROJECT / 'scripts/prepare_dense_visual_pilot.py'), '--raw-root', str(RAW_ROOT)], check=True)
import pandas as pd
inventory = pd.read_csv(TABLES / 'dense_window_image_inventory.csv', parse_dates=['timestamp'])
print('Unique raw files:', len(inventory), '| per-window spacing:', spacing_summary(inventory))
print('Stage 1 sheets:', FIGURES / 'dense_stage1')


## Human review
Select one seed and stage, then click **Open reviewer**. Frame navigation decodes only the selected crop. Blank states stay blank until you save them. Marking a pair records your human decision; Stage 2 requires adjacent available frames and cannot be saved before Stage 1.


In [ ]:
import ipywidgets as widgets
from IPython.display import display
seed_choice = widgets.Dropdown(options=[f'seed_{i:02d}' for i in range(1, 6)], description='Seed')
stage_choice = widgets.Dropdown(options=['stage1', 'stage2'], description='Stage')
open_button = widgets.Button(description='Open reviewer', button_style='primary')
viewer_output = widgets.Output()
def open_selected(_):
    with viewer_output:
        viewer_output.clear_output()
        review_widget(seed_choice.value, stage_choice.value, inventory,
                      TABLES / 'dense_visual_review_manifest.csv',
                      TABLES / 'refined_visual_events.csv')
open_button.on_click(open_selected)
display(widgets.VBox([widgets.HBox([seed_choice, stage_choice, open_button]), viewer_output]))


## Static fallback and final figures
If the interactive display is unavailable, use this button to save a contact sheet of all frames inside the human-saved Stage 1 interval. Review the PNG and use the CSV tables to enter human decisions. The final figure button works only after a valid Stage 2 pair is saved.


In [ ]:
fallback_button = widgets.Button(description='Save narrowed contact sheet')
figure_button = widgets.Button(description='Save final figure')
action_output = widgets.Output()
def save_narrowed(_):
    with action_output:
        action_output.clear_output()
        rows = prepare_stage2(seed_choice.value, inventory, TABLES / 'dense_visual_review_manifest.csv', TABLES / 'refined_visual_events.csv')
        destination = FIGURES / 'dense_stage2' / f'{seed_choice.value}_stage2.png'
        save_contact_sheet(rows, inventory, destination)
        print(destination)
def save_final(_):
    with action_output:
        action_output.clear_output()
        events = pd.read_csv(TABLES / 'refined_visual_events.csv', keep_default_na=False)
        row = events.loc[events.seed_id == seed_choice.value].iloc[0]
        rois = load_rois(TABLES / 'target_seed_rois.csv', TABLES / 'target_seed_frame_rois.csv')
        roi = rois.loc[rois.seed_id == seed_choice.value].iloc[0]
        bounds = tuple(int(roi[f'padded_{c}']) for c in ('x_min', 'y_min', 'x_max', 'y_max'))
        destination = FIGURES / 'refined_events' / f'{seed_choice.value}_visual_event.png'
        reviews = pd.read_csv(TABLES / 'dense_visual_review_manifest.csv', keep_default_na=False)
        save_event_figure(row, inventory, bounds, destination, reviews)
        print(destination)
fallback_button.on_click(save_narrowed)
figure_button.on_click(save_final)
display(widgets.VBox([widgets.HBox([fallback_button, figure_button]), action_output]))
